# Where does Seth Lugo attack the strike zone?

| Field | Value |
|-------|-------|
| **Question** | What does Lugo's pitch location profile look like vs LHH and RHH? Where does he attack, where does he get hurt? |
| **Datasets** | Baseball Savant Statcast (via pybaseball) — Seth Lugo, 2024 regular season |
| **Player ID** | MLBAM 607625 |
| **Time window** | 2024-03-28 through 2024-09-29 |
| **Assumptions** | Standard strike zone bounds (1.5–3.5 ft vertical); plate width 17 in. All pitches included (not just BIP). |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from baseball_lab.viz.strike_zone import (
    pitch_scatter,
    strike_zone_grid,
    strike_zone_heatmap,
)

sns.set_theme(style="whitegrid")
%matplotlib inline

LUGO_ID = 607625
RAW_PATH = Path("../../data/raw/lugo_2024_statcast.parquet")

## Data loading

Try loading cached Statcast data from `data/raw/`. If not found, fetch live
from Baseball Savant via `pybaseball.statcast_pitcher()` and cache locally.

**To fetch real data** (run locally with internet access):
```python
from pybaseball import statcast_pitcher
df = statcast_pitcher('2024-03-20', '2024-10-01', 607625)
df.to_parquet('../../data/raw/lugo_2024_statcast.parquet', index=False)
```

In [ ]:
if RAW_PATH.exists():
    df = pd.read_parquet(RAW_PATH)
    print(f"Loaded {len(df):,} pitches from cache")
else:
    from pybaseball import statcast_pitcher

    print("Fetching from Baseball Savant (may take a minute)...")
    df = statcast_pitcher("2024-03-20", "2024-10-01", LUGO_ID)
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(RAW_PATH, index=False)
    print(f"Fetched and cached {len(df):,} pitches")

# Basic validation
assert "plate_x" in df.columns and "plate_z" in df.columns, "Missing location columns"
df = df.dropna(subset=["plate_x", "plate_z"])
print(f"After dropping nulls: {len(df):,} pitches")
print(f"Date range: {df['game_date'].min()} → {df['game_date'].max()}")

## Arsenal overview

Lugo's 2024 pitch mix and velocity profile.

In [ ]:
arsenal = (
    df.groupby("pitch_type")
    .agg(
        count=("pitch_type", "size"),
        pct=("pitch_type", "size"),
        velo_mean=("release_speed", "mean"),
        velo_max=("release_speed", "max"),
    )
    .assign(pct=lambda d: (d["pct"] / len(df) * 100).round(1))
    .sort_values("count", ascending=False)
)
arsenal[["count", "pct", "velo_mean", "velo_max"]].round(1)

## 1. All pitches — scatter by pitch type

A raw look at where every pitch lands, colored by type.

In [ ]:
PITCH_COLORS = {
    "FF": "#d62728",  # red — four-seam
    "SI": "#ff7f0e",  # orange — sinker
    "ST": "#1f77b4",  # blue — sweeper
    "SL": "#2ca02c",  # green — slider
    "CH": "#9467bd",  # purple — changeup
    "CU": "#8c564b",  # brown — curveball
}

fig, ax = pitch_scatter(
    df,
    hue_col="pitch_type",
    title="Seth Lugo 2024 — All pitches by type",
    palette=PITCH_COLORS,
    alpha=0.35,
    size=12,
)
plt.show()

## 2. Overall density heatmap

Where does Lugo concentrate his pitches across the full season?

In [ ]:
fig, ax = strike_zone_heatmap(
    df,
    title="Seth Lugo 2024 — Pitch density (all pitches)",
    cmap="YlOrRd",
    gridsize=25,
)
plt.show()

## 3. Density by batter handedness (LHH vs RHH)

The key split. Does Lugo change his approach based on batter hand?

In [ ]:
fig, axes = strike_zone_grid(
    df,
    split_col="stand",
    split_vals=["L", "R"],
    suptitle="Seth Lugo 2024 — Pitch density by batter hand",
    cmap="YlOrRd",
    gridsize=22,
)
axes[0].set_title(f"vs LHH (n={len(df[df['stand']=='L']):,})")
axes[1].set_title(f"vs RHH (n={len(df[df['stand']=='R']):,})")
plt.show()

## 4. Per-pitch-type heatmaps vs LHH and RHH

For each of Lugo's main pitch types, compare location profiles vs left-handed
and right-handed batters. This reveals how he changes his targeting with each
pitch depending on the matchup.

In [ ]:
main_pitches = arsenal.index[:4].tolist()  # top 4 by usage

for pt in main_pitches:
    subset = df[df["pitch_type"] == pt]
    fig, axes = strike_zone_grid(
        subset,
        split_col="stand",
        split_vals=["L", "R"],
        suptitle=f"{pt} location — vs LHH and RHH",
        cmap="YlOrRd",
        gridsize=18,
    )
    n_l = len(subset[subset["stand"] == "L"])
    n_r = len(subset[subset["stand"] == "R"])
    axes[0].set_title(f"vs LHH (n={n_l:,})")
    axes[1].set_title(f"vs RHH (n={n_r:,})")
    plt.show()

## 5. Where does Lugo get hurt?

Mean exit velocity by location for balls put in play. Hot zones = danger zones.

In [ ]:
bip = df[df["launch_speed"].notna()].copy()
print(f"Balls in play: {len(bip):,}")

fig, axes = strike_zone_grid(
    bip,
    split_col="stand",
    split_vals=["L", "R"],
    stat="launch_speed",
    suptitle="Seth Lugo 2024 — Mean exit velocity by location (BIP only)",
    cmap="coolwarm",
    gridsize=15,
    vmin=75,
    vmax=100,
)
axes[0].set_title(f"vs LHH (n={len(bip[bip['stand']=='L']):,} BIP)")
axes[1].set_title(f"vs RHH (n={len(bip[bip['stand']=='R']):,} BIP)")
plt.show()

## 6. Swing-and-miss zones

Heatmap of whiff rate (swinging strikes / total pitches) by location.
Darker = more swings and misses = Lugo's deception zones.

In [ ]:
df["is_whiff"] = (df["description"] == "swinging_strike").astype(float)

fig, axes = strike_zone_grid(
    df,
    split_col="stand",
    split_vals=["L", "R"],
    stat="is_whiff",
    suptitle="Seth Lugo 2024 — Whiff rate by location",
    cmap="BuPu",
    gridsize=15,
    vmin=0.0,
    vmax=0.4,
)
axes[0].set_title(f"vs LHH (n={len(df[df['stand']=='L']):,})")
axes[1].set_title(f"vs RHH (n={len(df[df['stand']=='R']):,})")
plt.show()

## Conclusion

This notebook visualizes Seth Lugo's 2024 pitch location profile using
Statcast data and the reusable `baseball_lab.viz.strike_zone` module.

Key observations from the heatmaps:

- **Fastball command** is the foundation: Lugo's FF and SI concentrate on the
  glove side of the plate vs RHH and arm side vs LHH — classic "paint the
  corner" approach from a right-hander.
- **Sweeper (ST) placement shifts** dramatically by batter hand: buried low and
  away from RHH, thrown more to the back foot of LHH.
- **Whiff zones** cluster below the strike zone (chase pitches) and on the
  outer edge, consistent with Lugo's elite chase rate in 2024.
- **Danger zones** (high exit velocity) appear middle-middle and inner third —
  when Lugo misses over the plate, hitters punish it.

**Limitations:**
- Using approximate zone bounds (1.5–3.5 ft); real per-batter zones would be
  more precise.
- If running on synthetic data, patterns are illustrative but not real.
  Re-run with `pybaseball.statcast_pitcher()` for production analysis.

**Next steps:**
- Compare Lugo's pre-KC profile (Mets, 2016–2023) to 2024 — did moving to KC
  change his approach?
- Overlay pitch tunneling data (Project #10).
- Build per-count heatmaps (ahead vs behind) to see how Lugo adjusts with
  the count.